# 🧩 Bronze → Silver Pipeline (Delta Lake)

## 🎯 Objective
Build a simple data pipeline using Delta Lake to demonstrate the **Medallion Architecture**:

- Bronze = raw, messy data  
- Silver = cleaned, usable data  

---

## 🧠 Why this matters
In real-world data engineering:

- Data arrives messy (nulls, bad values, inconsistencies)
- We must clean and structure it before analytics
- This pipeline pattern is used in Databricks, Snowflake, and modern data platforms

In [0]:
# Create raw (bronze) data
data = [
    (1, "jenny", "US", 100),
    (2, "mike", "CA", None),
    (3, "sara", None, 200),
    (4, "john", "US", -50),
    (5, "anna", "UK", 300)
]

columns = ["user_id", "name", "country", "amount"]

df = spark.createDataFrame(data, columns)

df.write.mode("overwrite").format("delta").saveAsTable("bronze_users")

## 🛠 Step 1: Create Bronze Table (Raw Data)

### What I did
- Created a dataset with:
  - Null values
  - Missing fields
  - Invalid values (negative amounts)
- Wrote the data to a Delta table called `bronze_users`

### 💡 Why this matters
This simulates **real-world ingestion**, where:
- Data is often incomplete or incorrect
- Cleaning has NOT happened yet

👉 This is the **Bronze layer** — raw, messy data

In [0]:
%sql
SELECT * FROM bronze_users;

user_id,name,country,amount
1,jenny,US,100
2,mike,CA,null
3,sara,null,200
4,john,US,-50
5,anna,UK,300


## 🔍 Step 2: Inspect Bronze Data

### What I should notice
- Null values in `amount`
- Missing values in `country`
- Negative value in `amount`

### 💡 Why this matters
This reflects **real ingestion problems**:
- Data quality issues
- Incomplete records
- Invalid business values

👉 Bronze = **messy reality**

In [0]:
from pyspark.sql.functions import col, when

silver_df = (
    spark.table("bronze_users")
    .filter(col("amount") > 0)
    .withColumn("country", when(col("country").isNull(), "UNKNOWN").otherwise(col("country")))
)

silver_df.write.mode("overwrite").format("delta").saveAsTable("silver_users")

## 🧹 Step 3: Transform to Silver Layer

### What I did
- Removed invalid records (amount <= 0)
- Replaced missing country values with "UNKNOWN"

### 💡 Why this matters
This is the **data cleaning step**:
- Enforces business rules
- Improves data quality
- Prepares data for analytics

👉 Silver = **clean, usable data**

In [0]:
%sql
SELECT * FROM silver_users;

user_id,name,country,amount
1,jenny,US,100
3,sara,UNKNOWN,200
5,anna,UK,300


## ✅ Step 4: Inspect Silver Data

### What changed
- Negative values removed
- Null country values handled
- Dataset is now consistent

### 💡 Why this matters
The data is now:
- Reliable
- Query-ready
- Suitable for downstream analytics

👉 This is what analysts and dashboards use

## 🚀 Final Takeaways

### 🧠 Key Concepts Demonstrated
- Bronze layer captures raw, unprocessed ingestion data  
- Silver layer applies data quality rules and transformations  
- Delta Lake provides reliable, scalable table storage  

---

### ⚙️ Pipeline Summary
This notebook implements a simple **Bronze → Silver pipeline** using Delta Lake:

- Raw ingestion data is stored in a Bronze table  
- Data is cleaned and validated into a Silver table  
- Transformations enforce business rules and improve data quality  

---

### 💡 Why This Matters
This pattern is foundational in modern data engineering:

- Separates raw vs curated data layers  
- Improves data reliability and usability  
- Supports scalable, maintainable data pipelines  

---

### 🔍 Key Insight
Effective data engineering is not just about transforming data —  
it’s about structuring data systems so that quality, clarity, and scalability are built in from the start.